# 🚀 Máy Chủ Dịch Thuật Ollama (Qwen2.5:7B) Trên Google Colab
Notebook này thiết lập máy chủ dịch thuật **Ollama** sử dụng GPU miễn phí trên Google Colab và mở đường hầm **Cloudflare Tunnel** công khai để máy Local kết nối và điều phối dịch tự động.

### ⚠️ HƯỚNG DẪN TRƯỚC KHI CHẠY:
1. Trên thanh menu Colab, chọn **Thời gian chạy (Runtime)** ➔ **Thay đổi loại thời gian chạy (Change runtime type)**.
2. Tại mục **Bộ tăng tốc phần cứng (Hardware accelerator)**: Chọn **T4 GPU** rồi bấm **Lưu (Save)**.
3. Bấm nút **Chạy lần lượt** từng Cell bên dưới từ trên xuống dưới.

In [ ]:
# Bước 1: Kiểm tra xem Colab đã nhận GPU T4 chưa
!nvidia-smi

In [ ]:
# Bước 2: Cài đặt gói nén zstd, Ollama và Cloudflared
# Dọn dẹp tiến trình cũ nếu có
!pkill -9 -f ollama || true
!pkill -9 -f cloudflared || true

!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
# Bước 3: Khởi động Ollama ngầm với quyền truy cập mở & Tải mô hình Qwen2.5 7B
import subprocess
import time
import os

# Dọn tiến trình cũ để tránh xung đột cổng 11434
!pkill -9 -f ollama || true
time.sleep(1)

# Cấu hình cho phép mọi nguồn và IP kết nối qua Tunnel (Tránh lỗi 403 Forbidden)
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["OLLAMA_ORIGINS"] = "*"

print("🚀 Đang khởi động service Ollama ngầm...")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env)
time.sleep(4)

print("📥 Đang tải mô hình qwen2.5:7b-instruct (dung lượng ~4.7 GB, mất khoảng 1-2 phút)...\n")
!ollama pull qwen2.5:7b-instruct

In [ ]:
# Bước 4: Mở Cloudflare Tunnel và Tự động Test kết nối 200 OK
import subprocess
import re
import time
import requests

# Dọn dẹp tunnel cũ nếu có
!pkill -9 -f cloudflared || true
time.sleep(1)

print("🌐 Đang mở Cloudflare Tunnel công khai...")
# Thêm --http-host-header localhost để vượt qua kiểm tra bảo mật host của Ollama
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434", "--http-host-header", "localhost"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print(f"[*] Đã tạo đường hầm: {public_url}")
    print("[*] Đang chạy kiểm thử kết nối nội bộ...")
    time.sleep(3)
    try:
        test_resp = requests.get(f"{public_url}/api/tags", timeout=15)
        if test_resp.status_code == 200:
            print("\n" + "=" * 65)
            print("🎉 MÁY CHỦ DỊCH THUẬT OLLAMA ĐÃ SẴN SÀNG (ĐÃ TEST HTTP 200 OK)!")
            print(f"👉 URL KẾT NỐI (Copy đường link bên dưới):")
            print(f"   {public_url}")
            print("=" * 65)
            print("\n💡 Các bước tiếp theo trên máy Local:")
            print("1. Mở terminal máy Local chạy: python run_cli.py")
            print("2. Chọn [1] Dịch thuật AI ➔ Chọn [1] Dịch tiểu thuyết raw")
            print("3. Dán link trên vào và bắt đầu dịch tự động!")
            print("\n⚠️ LƯU Ý: Giữ ô này tiếp tục chạy (không tắt tab) trong lúc máy Local đang dịch.\n")
        else:
            print(f"⚠️ Cảnh báo phản hồi: HTTP {test_resp.status_code}. Nội dung: {test_resp.text}")
    except Exception as e:
        print(f"⚠️ Lỗi kiểm thử kết nối: {e}")

try:
    tunnel_proc.wait()
except KeyboardInterrupt:
    print("\n🛑 Đang dừng máy chủ...")
    tunnel_proc.kill()
    ollama_proc.kill()